In [1]:
import os
from pathlib import Path
from typing import List

from tqdm import tqdm
import numpy as np
import pandas as pd
try:
    import google.generativeai as genai
except ImportError as exc:
    raise ImportError("Install google-generativeai via `pip install google-generativeai`.") from exc


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ---- Configuration ----
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
if GOOGLE_API_KEY is not None:
    GEMINI_API_KEY = GOOGLE_API_KEY
if not GEMINI_API_KEY:
    raise EnvironmentError("Set GEMINI_API_KEY in your environment before running this cell.")

genai.configure(api_key=GEMINI_API_KEY)

ROOT_DIR = Path("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/outputs/prompt & rag/20251028_085918 - Run/generated_code-v2").expanduser()
if not ROOT_DIR.exists():
    raise FileNotFoundError(f"Root directory not found: {ROOT_DIR}")

MODEL_NAME = "text-embedding-004"
BATCH_SIZE = 250  # number of .py files to embed before logging progress


In [5]:
def read_python_file(file_path: Path) -> str:
    """Return the contents of a Python file with a lightweight header."""
    return f"# File: {file_path.name}\n" + file_path.read_text(encoding="utf-8", errors="ignore")


def embed_text(texts: List[str]) -> List[float]:
    response = genai.embed_content(
        model=MODEL_NAME,
        content=texts,
        task_type="SEMANTIC_SIMILARITY",
    )
    return response["embedding"]


def embed_python_file(file_path: Path) -> np.ndarray:
    source = read_python_file(file_path)
    if not source.strip():
        raise ValueError(f"{file_path} is empty or unreadable")
    return np.asarray(embed_text(source), dtype=np.float32)


In [6]:
records = []
pattern_dirs = [p for p in sorted(ROOT_DIR.iterdir()) if p.is_dir()]
if not pattern_dirs:
    raise ValueError(f"No pattern directories detected in {ROOT_DIR}")

python_files = []
for pattern_dir in pattern_dirs:
    python_files.extend((pattern_dir.name, py_file) for py_file in sorted(pattern_dir.glob("**/*.py")))

if not python_files:
    raise ValueError("No .py files found under the supplied root directory.")

total_files = len(python_files)
for batch_start in range(0, total_files, BATCH_SIZE):
    batch = python_files[batch_start : batch_start + BATCH_SIZE]
    print(
        f"Processing files {batch_start + 1}-{batch_start + len(batch)} / {total_files}"
    )

    texts = [read_python_file(py_path) for _, py_path in batch]
    embeddings = embed_text(texts)
    for (pattern_name, py_path), embedding_vector in zip(batch, embeddings):
        records.append(
            {
                "pattern": pattern_name,
                "file": str(py_path.relative_to(ROOT_DIR)),
                "embedding": embedding_vector,
            }
        )
    
    # for pattern_name, py_path in tqdm(batch, desc="Files", leave=False):
    #     try:
    #         embedding_vector = embed_python_file(py_path)
    #     except ValueError as err:
    #         print(f"Skipping {py_path}: {err}")
    #         continue
    #     records.append(
    #         {
    #             "pattern": pattern_name,
    #             "file": str(py_path.relative_to(ROOT_DIR)),
    #             "embedding": embedding_vector,
    #         }
    #     )

if not records:
    raise ValueError("No embeddings were generated; ensure .py files have content.")

embedding_length = len(records[0]["embedding"])
rows = []
for record in records:
    row = {f"dim_{i+1}": value for i, value in enumerate(record["embedding"])}
    row["pattern"] = record["pattern"]
    row["file"] = record["file"]
    rows.append(row)

embeddings_df = pd.DataFrame(rows)
print(
    f"Generated embeddings for {len(embeddings_df)} files across {len(pattern_dirs)} patterns with {embedding_length} dimensions."
)
display(embeddings_df.head())

OUTPUT_PATH = Path("./results/pattern_embeddings/gemini_pattern_embedding_v3.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
embeddings_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved embeddings to {OUTPUT_PATH.resolve()}")


Processing files 1-250 / 2081
Processing files 251-500 / 2081
Processing files 501-750 / 2081
Processing files 751-1000 / 2081
Processing files 1001-1250 / 2081
Processing files 1251-1500 / 2081
Processing files 1501-1750 / 2081
Processing files 1751-2000 / 2081
Processing files 2001-2081 / 2081
Generated embeddings for 2081 files across 22 patterns with 768 dimensions.


,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10,...,dim_761,dim_762,dim_763,dim_764,dim_765,dim_766,dim_767,dim_768,pattern,file
0,0.008586,-0.000224,-0.073952,0.040551,0.041076,0.051949,0.053746,0.020028,0.009389,0.006378,...,-0.029720,0.062391,0.007221,0.005423,-0.022622,-0.054182,0.068976,-0.041483,Advanced LLM Prompting,Advanced LLM Prompting/AIlice-cluster_37.py
1,0.025723,0.010738,-0.067647,0.041301,0.041939,0.043754,0.024987,0.018802,0.020988,-0.032886,...,-0.000288,0.062254,0.007914,0.032241,-0.018615,-0.059511,0.072126,-0.037370,Advanced LLM Prompting,Advanced LLM Prompting/EvoAgentX-cluster_85.py
2,-0.028630,0.014242,-0.084164,0.006628,0.054243,0.067711,0.027371,-0.006613,-0.034239,-0.014033,...,0.016043,0.042599,0.010227,0.028914,-0.026917,-0.046186,0.094621,-0.040274,Advanced LLM Prompting,Advanced LLM Prompting/pattern_1.py
3,-0.005000,0.003826,-0.077749,-0.017940,0.028686,0.041942,0.004677,0.013499,-0.024628,-0.017825,...,0.028125,0.043700,-0.006213,0.003327,-0.021241,0.000565,0.114473,-0.067262,Advanced LLM Prompting,Advanced LLM Prompting/pattern_10.py
4,-0.030237,-0.001334,-0.048906,0.036134,0.053802,0.039611,0.050056,-0.000391,-0.026402,-0.024241,...,-0.022873,0.049713,0.004830,-0.002995,-0.079745,-0.064036,0.085545,-0.041079,Advanced LLM Prompting,Advanced LLM Prompting/pattern_11.py


Saved embeddings to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_pattern_embedding_v3.csv


In [ ]:
embeddings_df.shape